# 03 - Treinamento LightGBM + Optuna + MLflow
## Home Credit Default Risk

**Dataset:** 307,511 linhas × 1,042 features | Taxa de default: 8.07%

Pipeline:
1. Tuning rápido com Optuna (20 trials, 3-fold)
2. Treinamento final com 5-Fold StratifiedKFold
3. Tracking de experimentos com MLflow
4. Geração do `submission.csv`

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import mlflow
import mlflow.lightgbm
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

PROC_DIR   = '../data/processed'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

SEED    = 42
N_FOLDS = 5
print('Setup OK!')

## 1. Carregar e Preparar Dados

In [ ]:
train = pd.read_parquet(f'{PROC_DIR}/train_processed.parquet')
test  = pd.read_parquet(f'{PROC_DIR}/test_processed.parquet')

# Sanitiza nomes de colunas — parquets gerados antes do fix podem ter caracteres especiais
train.columns = train.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)
test.columns  = test.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

# ── Label Encoding para colunas object que sobraram (alta cardinalidade) ──
# LightGBM exige int/float/bool — não aceita strings
obj_cols = train.select_dtypes('object').columns.tolist()
if obj_cols:
    print(f'Aplicando Label Encoding em {len(obj_cols)} colunas: {obj_cols}')
    le = LabelEncoder()
    for col in obj_cols:
        # Treina o encoder no train+test combinados para cobrir todas as categorias
        all_vals = pd.concat([train[col], test[col] if col in test.columns else pd.Series(dtype=str)]).fillna('__NaN__')
        le.fit(all_vals.astype(str))
        train[col] = le.transform(train[col].fillna('__NaN__').astype(str)).astype(np.int16)
        if col in test.columns:
            test[col] = le.transform(test[col].fillna('__NaN__').astype(str)).astype(np.int16)
else:
    print('Nenhuma coluna object encontrada — tudo OK!')

TARGET    = 'TARGET'
ID_COL    = 'SK_ID_CURR'
FEAT_COLS = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[FEAT_COLS]
y      = train[TARGET]
X_test = test[[c for c in FEAT_COLS if c in test.columns]]

# Alinha colunas do test com o train (segurança extra)
for col in set(FEAT_COLS) - set(X_test.columns):
    X_test[col] = 0
X_test = X_test[FEAT_COLS]

# Scale pos weight (compensa o desbalanceamento ~92/8)
SPW = (y == 0).sum() / (y == 1).sum()

print(f'\nFeatures: {len(FEAT_COLS):,}')
print(f'Train: {X.shape} | Test: {X_test.shape}')
print(f'Taxa de default: {y.mean():.2%}')
print(f'scale_pos_weight: {SPW:.2f}')

# Verificação final — nenhuma coluna object deve restar
bad = X.select_dtypes('object').columns.tolist()
assert not bad, f'Colunas object ainda presentes: {bad}'
print('Verificação de tipos: OK ✓')

In [ ]:
import re

# Diagnóstico: quais colunas ainda têm caracteres especiais?
bad_cols = [col for col in X.columns if re.search(r'[^A-Za-z0-9_]', col)]
print(f'Colunas com caracteres especiais em X: {len(bad_cols)}')
if bad_cols:
    print('Exemplos:', bad_cols[:10])
    # Sanitiza de forma explícita usando re.sub
    X.columns        = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X.columns]
    X_test.columns   = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_test.columns]
    FEAT_COLS        = list(X.columns)
    bad_after = [col for col in X.columns if re.search(r'[^A-Za-z0-9_]', col)]
    print(f'Após sanitização: {len(bad_after)} colunas ruins restantes')
else:
    print('Nomes de colunas OK — nenhum caractere especial encontrado.')

## 2. Tuning com Optuna (20 trials × 3-fold)

In [ ]:
def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbosity':         -1,
        'boosting_type':     'gbdt',
        'random_state':      SEED,
        'n_estimators':      500,
        'scale_pos_weight':  SPW,
        'learning_rate':     trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 300),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 0.9),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 0.9),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(30, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)

print('Iniciando Optuna (20 trials × 3-fold)...')
study = optuna.create_study(direction='maximize', study_name='lgbm_hcdr')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'\nMelhor AUC (tuning 3-fold): {study.best_value:.5f}')
print('Melhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 3. Treinamento Final com 5-Fold + MLflow

In [ ]:
print("=" * 60)
print(f"OOF AUC: {oof_auc:.5f}")
print(f"Std entre folds: {np.std(fold_aucs):.5f}")
print("-" * 60)
print("Kaggle submission description:")
print(f'LightGBM 5-fold CV with Optuna hyperparameter tuning. Features engineered from all auxiliary tables (bureau, previous applications, POS cash, installments, credit card). OOF AUC: {oof_auc:.5f}.')
print("=" * 60)

In [ ]:
BEST_PARAMS = {
    'objective':        'binary',
    'metric':           'auc',
    'verbosity':        -1,
    'boosting_type':    'gbdt',
    'random_state':     SEED,
    'n_estimators':     2000,
    'scale_pos_weight': SPW,
    **study.best_params
}

mlflow.set_experiment('home-credit-default-risk')

with mlflow.start_run(run_name='lgbm_5fold_optuna'):
    mlflow.log_params({k: v for k, v in BEST_PARAMS.items()
                       if k not in ['objective', 'metric', 'verbosity']})
    mlflow.log_param('n_features', len(FEAT_COLS))

    skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs  = []
    models     = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f'\n─── Fold {fold+1}/{N_FOLDS} ───')
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**BEST_PARAMS)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(200)
            ]
        )

        oof_preds[val_idx]  = model.predict_proba(X_val)[:, 1]
        test_preds         += model.predict_proba(X_test)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_aucs.append(fold_auc)
        models.append(model)
        print(f'Fold {fold+1} AUC: {fold_auc:.5f}')

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n{"="*45}')
    print(f'OOF AUC FINAL : {oof_auc:.5f}')
    print(f'Std por fold  : {np.std(fold_aucs):.5f}')
    print(f'{"="*45}')

    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('std_fold_auc', np.std(fold_aucs))
    for i, auc in enumerate(fold_aucs):
        mlflow.log_metric(f'fold_{i+1}_auc', auc)

## 4. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature':    FEAT_COLS,
    'importance': np.mean([m.feature_importances_ for m in models], axis=0)
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
top40 = importances.head(40)
colors = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(top40))]
ax.barh(top40['feature'][::-1], top40['importance'][::-1],
        color=colors[::-1], edgecolor='white')
ax.set_title(f'Top 40 Features por Importância (LightGBM)\nOOF AUC: {oof_auc:.5f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importância média entre folds')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print(importances.head(20).to_string(index=False))

## 5. Gerar Submissão

In [ ]:
submission = pd.DataFrame({
    'SK_ID_CURR': test[ID_COL].astype(int),
    'TARGET':     test_preds
})

submission.to_csv('../submission.csv', index=False)
print('submission.csv gerado!')
print(f'Shape: {submission.shape}')
print(f'Predições — min: {test_preds.min():.4f} | média: {test_preds.mean():.4f} | max: {test_preds.max():.4f}')
submission.head(10)